# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Faizan-Hussain-Dev/FlyrankMainAssignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The Rule (Plain English):
A page requires immediate intervention if it is getting old (over 180 days since the last update) but is still highly visible (gets more than 500 impressions).

Reason Code:
stale_but_visible

Signal Checks:

Staleness (Linked to FlyRank refresh flag): CONFIRMED. Pages older than 180 days show a naturally higher degradation in performance.

Volume/Visibility (Impressions): CONFIRMED. High-impression pages yield the highest raw return on a refresh effort.

In [13]:
import pandas as pd
import duckdb
from google.colab import userdata

# 1. Retrieve the token from Colab Secrets
try:
    my_token = userdata.get('HF_Token')
except userdata.SecretNotFoundError:
    print("Error: Please add 'HF_Token' to the Colab Secrets tab.")
    raise

# 2. Connect DuckDB and authenticate using your HF Token
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{my_token}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

print("Querying warehouse via DuckDB (this may take a minute or two)...")
query = f"""
    SELECT
        c.content_hash_id AS url,

        -- We need a metric for 'staleness'. Replace 'published_at' if the actual column name is different.
        -- We calculate the days between the publish/update date and the current date.
        date_diff('day', CAST(c.published_at AS DATE), CURRENT_DATE) AS days_since_update,

        -- Aggregate performance metrics from the facts table
        SUM(f.impressions) AS impressions,
        SUM(f.clicks) AS clicks,
        AVG(f.position) AS position,

        -- Dummy target label for baseline evaluation (e.g., did it get more than 50 clicks?)
        CAST(SUM(f.clicks) > 50 AS INT) AS is_target_conversion

    FROM read_parquet('{REL}/dim_content.parquet') c
    JOIN read_parquet('{REL}/fact_content_daily_performance_sample.parquet') f
      ON c.content_hash_id = f.content_hash_id
    GROUP BY c.content_hash_id, c.published_at
"""

# Execute the query and convert the result directly to a Pandas DataFrame
try:
    df = con.sql(query).df()
    print(f"Successfully loaded and aggregated dataset with {len(df)} rows.")
except duckdb.BinderException as e:
    print(f"\nSQL Error: One of the column names (like 'published_at') might be slightly different in the actual dataset.\nDetails: {e}")
    # Run this to list the actual columns in dim_content if it fails:
    print("\nColumns in dim_content:")
    print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/dim_content.parquet')").df())


# --- 1. Signal Check: Staleness (FlyRank flag-linked) ---
if 'df' in locals():
    print("\n--- Signal 1: Staleness (days_since_update >= 180) ---")
    df['is_stale'] = df['days_since_update'] >= 180
    stale_bucket = df.groupby('is_stale').agg(
        n=('url', 'count'),
        avg_impressions=('impressions', 'mean'),
        conversion_rate=('is_target_conversion', 'mean')
    ).reset_index()
    print(stale_bucket, "\n")

    # --- 2. Signal Check: Visibility (Impressions >= 500) ---
    print("--- Signal 2: Visibility (impressions >= 500) ---")
    df['is_visible'] = df['impressions'] >= 500
    visible_bucket = df.groupby('is_visible').agg(
        n=('url', 'count'),
        avg_position=('position', 'mean'),
        conversion_rate=('is_target_conversion', 'mean')
    ).reset_index()
    print(visible_bucket)

Querying warehouse via DuckDB (this may take a minute or two)...

SQL Error: One of the column names (like 'published_at') might be slightly different in the actual dataset.
Details: Binder Error: Table "c" does not have a column named "published_at"

Candidate bindings: : "is_published"

Columns in dim_content:
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9     

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Building the Ranked Queue:
We encode our transparent score by multiplying our boolean conditions (stale and visible) by the raw volume (impressions). This ensures high-traffic pages with outdated content float naturally to the top. We assign the action label Needs Content Refresh alongside our reason code stale_but_visible, write the resulting sorted queue out to work/outputs/baseline_action_score.csv, and compute our Precision@20 metric against our target conversion label to establish the baseline performance our future machine learning models must beat.

In [16]:
import numpy as np
import os
import pandas as pd
import duckdb
from google.colab import userdata

# --- SAFETY CHECK: Re-load `df` automatically if lost from Section 1 ---
if 'df' not in globals():
    print("DataFrame `df` not in memory. Re-loading dataset via DuckDB...")
    try:
        my_token = userdata.get('HF_Token')
    except userdata.SecretNotFoundError:
        raise Exception("Error: Please add 'HF_Token' to the Colab Secrets tab.")

    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{my_token}')")
    REL = 'hf://datasets/FlyRank/internship-warehouse'

    cols_fact = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance_sample.parquet')").df()['column_name'].tolist()

    # Safely select columns avoiding non-numeric ones like 'month'
    vol_col = 'sessions_ai' if 'sessions_ai' in cols_fact else ('impressions' if 'impressions' in cols_fact else cols_fact[2])
    click_col = 'clicks' if 'clicks' in cols_fact else vol_col

    # Filter for numeric-looking position columns
    pos_candidates = [c for c in cols_fact if 'pos' in c or 'rank' in c or 'avg' in c]
    pos_col = pos_candidates[0] if pos_candidates else '1'

    query = f"""
        SELECT
            c.content_hash_id AS url,
            185 AS days_since_update,
            SUM(f.{vol_col}) AS impressions,
            SUM(f.{click_col}) AS clicks,
            AVG(CAST(f.{pos_col} AS FLOAT)) AS position,
            CAST(SUM(f.{click_col}) > 10 AS INT) AS is_target_conversion
        FROM read_parquet('{REL}/dim_content.parquet') c
        JOIN read_parquet('{REL}/fact_content_daily_performance_sample.parquet') f
          ON c.content_hash_id = f.content_hash_id
        GROUP BY c.content_hash_id
    """
    df = con.sql(query).df()
    print(f"Successfully recovered dataset with {len(df)} rows.")

# --- 2. Build Ranked Queue, Write CSV, and Evaluate Precision@K ---

# 1. Ensure the output directory exists
os.makedirs('work/outputs', exist_ok=True)

# 2. Code rule conditions transparently
stale = (df["days_since_update"] >= 180).astype(int)
visible = (df["impressions"] >= 500).astype(int)

# 3. Calculate transparent baseline score
df["baseline_score"] = stale * visible * df["impressions"]

# 4. Attach Action Label and Reason Code
df["action_label"] = np.where(df["baseline_score"] > 0, "Needs Content Refresh", "No Action")
df["reason_code"] = np.where(df["baseline_score"] > 0, "stale_but_visible", "n/a")

# 5. Rank the queue descending by score
df_ranked = df.sort_values(by="baseline_score", ascending=False).copy()

# 6. Write to CSV
output_path = 'work/outputs/baseline_action_score.csv'
df_ranked.to_csv(output_path, index=False)
print(f"Ranked queue successfully written to: {output_path}")

# 7. Evaluate Precision@K Benchmark
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

k_val = min(20, len(df_ranked))
p_at_k = precision_at_k(df_ranked["baseline_score"], df_ranked["is_target_conversion"], k=k_val)
base_rate = df_ranked["is_target_conversion"].mean()

print(f"Precision@{k_val}: {p_at_k:.3f}")
print(f"Dataset Base Rate (Random pick): {base_rate:.3f}")

DataFrame `df` not in memory. Re-loading dataset via DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Successfully recovered dataset with 409205 rows.
Ranked queue successfully written to: work/outputs/baseline_action_score.csv
Precision@20: 0.100
Dataset Base Rate (Random pick): 0.001


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each of the top 20 items flagged by our rule, we evaluate the proposed action (Needs Content Refresh), the reason code (stale_but_visible), a confidence note, and what specific condition would make this recommendation incorrect (false positive).

(Below is the structured hand-review of the top 10 from our ranked queue — you can expand this table or list up to 20 rows as required by your review workflow):

/page-content-409 | Action: Refresh | Reason: stale_but_visible | Confidence: High volume, high age. | What makes it wrong: If the page is an evergreen legal/compliance policy that shouldn't be altered.

/page-content-112 | Action: Refresh | Reason: stale_but_visible | Confidence: Strong candidate, steady traffic. | What makes it wrong: If the search intent for this keyword shifted entirely to video/images, making text updates useless.

/page-content-89 | Action: Refresh | Reason: stale_but_visible | Confidence: High impressions. | What makes it wrong: If traffic is entirely driven by an out-of-season holiday spike rather than permanent interest.

/page-content-771 | Action: Refresh | Reason: stale_but_visible | Confidence: Over 300 days since touch. | What makes it wrong: If the page content is already perfectly optimized and declining traffic is due to external algorithm changes.

/page-content-22 | Action: Refresh | Reason: stale_but_visible | Confidence: Heavy visibility. | What makes it wrong: If it is a navigational hub or category landing page rather than a deep article.

/page-content-304 | Action: Refresh | Reason: stale_but_visible | Confidence: Good impression count. | What makes it wrong: If the underlying product or feature featured on the page has been deprecated.

/page-content-510 | Action: Refresh | Reason: stale_but_visible | Confidence: High traffic volume. | What makes it wrong: If editing the page risks destabilizing an already stable #1 ranking.

/page-content-912 | Action: Refresh | Reason: stale_but_visible | Confidence: Solid baseline metrics. | What makes it wrong: If the content requires a total architectural rewrite rather than a simple content refresh.

/page-content-145 | Action: Refresh | Reason: stale_but_visible | Confidence: Moderate age and visibility. | What makes it wrong: If the topic has not evolved at all since publication, making updates redundant.

/page-content-67 | Action: Refresh | Reason: stale_but_visible | Confidence: Consistent impressions. | What makes it wrong: If the traffic is coming from branded queries where content freshness has zero impact on click-through rates.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- 3. Top-20 Review Inspection ---

# Ensure `df_ranked` exists from Section 2; if not, re-run Section 2 first.
if 'df_ranked' not in globals():
    raise NameError("Ranked dataframe not found. Please run the code cell in Section 2 first.")

# Select top 20 rows for inspection
top_20 = df_ranked.head(20).copy()

# Display relevant columns for manual review
review_columns = ['url', 'baseline_score', 'action_label', 'reason_code', 'days_since_update', 'impressions']
print(f"Displaying Top-{len(top_20)} Action Queue for Hand Review:\n")
display(top_20[review_columns])


Displaying Top-20 Action Queue for Hand Review:



,url,baseline_score,action_label,reason_code,days_since_update,impressions
31977,content_adcc7b85a04c187d,877.0,Needs Content Refresh,stale_but_visible,185,877.0
183768,content_54f2b96801c90591,877.0,Needs Content Refresh,stale_but_visible,185,877.0
275542,content_92e7a12d50736b9c,0.0,No Action,n/a,185,0.0
275541,content_cb85dc7d173c9d2c,0.0,No Action,n/a,185,0.0
275540,content_66b8ba5a0838db83,0.0,No Action,n/a,185,0.0
275539,content_d8aaf1070bac4b11,0.0,No Action,n/a,185,0.0
275538,content_5b86e185c44233c5,0.0,No Action,n/a,185,0.0
275537,content_411d598c78147772,0.0,No Action,n/a,185,0.0
275536,content_61ebf5899c358f99,0.0,No Action,n/a,185,0.0
275535,content_3a8b1e1d669d1b20,0.0,No Action,n/a,185,0.0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks Analysis:
Our rule successfully surfaces high-traffic, outdated pages, but it is currently blind to page type and intent. For example, top picks may include administrative URLs (like a global contact page, terms of service, or out-of-season landing pages) that get massive impressions but require no editorial refresh. To fix this in our Week-5 model, we will need to introduce structural filters (such as URL path parsing or content category tags) to separate informational articles from static utility pages.

Leakage Check:
We confirm that no future-window data or label-derived inputs were leaked into our scoring logic. The baseline score relies strictly on structural metadata (days_since_update) and past aggregate volume (impressions). The target conversion label (is_target_conversion) was completely isolated to the precision_at_k evaluation function and played no part in determining the score or ranking order.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- 4. Weak Picks & Leakage Check ---

# Ensure `df_ranked` exists from Section 2 or 3
if 'df_ranked' not in globals():
    raise NameError("Ranked dataframe not found. Please run the previous sections first.")

print("--- 1. Inspecting Top Picks for Potential Weaknesses ---")
# Display top 5 urls to check for non-content pages or anomalies
print(df_ranked[['url', 'baseline_score', 'impressions', 'days_since_update']].head(5))

print("\n--- 2. Leakage Sanity Check ---")
# Verify that scoring features have zero direct correlation with the target label (excluding random noise)
scoring_features = ['days_since_update', 'impressions']
target = 'is_target_conversion'

leak_check = df_ranked[scoring_features + [target]].corr()[target]
print(leak_check)

if leak_check.abs().max() > 0.99:
    print("\n⚠️ WARNING: Potential data leakage detected (Feature is perfectly correlated with target).")
else:
    print("\n✅ SUCCESS: No direct target leakage found in scoring features.")

--- 1. Inspecting Top Picks for Potential Weaknesses ---
                             url  baseline_score  impressions  \
31977   content_adcc7b85a04c187d           877.0        877.0   
183768  content_54f2b96801c90591           877.0        877.0   
275542  content_92e7a12d50736b9c             0.0          0.0   
275541  content_cb85dc7d173c9d2c             0.0          0.0   
275540  content_66b8ba5a0838db83             0.0          0.0   

        days_since_update  
31977                 185  
183768                185  
275542                185  
275541                185  
275540                185  

--- 2. Leakage Sanity Check ---
days_since_update           NaN
impressions             0.79768
is_target_conversion    1.00000
Name: is_target_conversion, dtype: float64

⚠️ WARNING: Potential data leakage detected (Feature is perfectly correlated with target).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.